In [1]:
import pandas as pd
dataset = pd.read_excel("C:/Users/bandarua/crda_gp/data/CuTHQ3.xlsx")
dataset = dataset.drop(columns="Experiment")
dataset

,en (mL),EtOH (mL),IPA (mL),DMF,Rxn time (min),Rxn temp (cels),1/R,yield
0,0.046,0,0,0,10,120,0.087413,0.2160
1,0.046,0,0,0,20,120,0.097371,0.2180
2,0.046,0,0,0,30,120,0.128535,0.1190
3,0.046,0,0,0,40,120,0.110011,0.1240
4,0.046,0,0,0,10,140,0.081766,0.2200
5,0.046,0,0,0,20,140,0.088339,0.1810
6,0.046,0,0,0,30,140,0.102669,0.1280
7,0.046,0,0,0,40,140,0.121065,0.0980
8,0.046,4,0,0,10,120,0.088261,0.0540
9,0.046,4,0,0,20,120,0.086730,0.0550


In [5]:
from sklearn.model_selection import train_test_split
import torch
import numpy as np
from crda_gp.utils import set_seed

set_seed(42)

X = np.array(dataset.drop(columns=["1/R", "yield"]))
y_1 = np.array(dataset["1/R"])
y_2 = np.array(dataset["yield"])

X_train, X_test, y_train_1, y_test_1 = train_test_split(X, y_1, test_size=0.1, random_state=42)

# Turn data splits into tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_1_tensor = torch.tensor(y_train_1, dtype=torch.float32)
y_test_1_tensor = torch.tensor(y_test_1, dtype=torch.float32)

In [6]:
import gpytorch
from crda_gp.models import ARDExactGPModel_Dense
set_seed(42)
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ARDExactGPModel_Dense(train_x=X_train_tensor,
                              train_y=y_train_1_tensor,
                              likelihood=likelihood)
model

ARDExactGPModel_Dense(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): ConstantMean()
  (covar_module): ScaleKernel(
    (base_kernel): RBFKernel(
      (raw_lengthscale_constraint): Positive()
    )
    (raw_outputscale_constraint): Positive()
  )
)

In [7]:
from crda_gp.utils import train_step
set_seed(42)
train_step(X_train=X_train_tensor,
           y_train=y_train_1_tensor,
           model=model,
           likelihood=likelihood)

Iter: 10/100 | Loss: 0.706 | Noise: 0.312
Iter: 20/100 | Loss: 0.231 | Noise: 0.122
Iter: 30/100 | Loss: -0.274 | Noise: 0.044
Iter: 40/100 | Loss: -0.749 | Noise: 0.015
Iter: 50/100 | Loss: -1.091 | Noise: 0.006
Iter: 60/100 | Loss: -1.218 | Noise: 0.003
Iter: 70/100 | Loss: -1.225 | Noise: 0.002
Iter: 80/100 | Loss: -1.271 | Noise: 0.003
Iter: 90/100 | Loss: -1.308 | Noise: 0.003
Iter: 100/100 | Loss: -1.328 | Noise: 0.002


In [8]:
from crda_gp.utils import val_step
set_seed(42)
y_pred = val_step(X_test=X_test_tensor,
                  model=model,
                  likelihood=likelihood)

In [9]:
from sklearn.metrics import mean_squared_error
set_seed(42)
MSE = mean_squared_error(y_true=y_test_1_tensor, 
                         y_pred=y_pred)

In [10]:
MSE

0.002021840075030923

In [15]:
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    pred_dist = likelihood(model(X_train_tensor))
    y_new_aug = pred_dist.mean

residual = y_train_1_tensor - y_new_aug
corr = abs(pd.DataFrame(X_train).corrwith(pd.Series(residual)))

preturb_idx= corr[abs(corr) < corr.mean().item()].sort_values(ascending=False).index.values
preturb_idx

c:\Users\bandarua\crda_gp\env\Lib\site-packages\gpytorch\models\exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(


array([5, 1, 4])

In [16]:
corr.mean().item()

0.27523828115103205

In [17]:
from crda_gp.crda import CRDA_function
X_train_CRDA_tensor, y_train_1_CRDA_tensor = CRDA_function(X_train_tensor,
                                                           y_train_1_tensor,
                                                           model=model,
                                                           likelihood=likelihood,
                                                           perturb_idx=preturb_idx,
                                                           p=0.1)

Starting the Loop
Iter: 0
Iter: 1
Iter: 2
Iter: 3
Iter: 4
Iter: 5
Iter: 6
Iter: 7
Iter: 8
Iter: 9
Iter: 10
Iter: 11
Iter: 12
Iter: 13
Iter: 14
Iter: 15
Iter: 16
Iter: 17
Iter: 18
Iter: 19
Iter: 20
Iter: 21
Iter: 22
Iter: 23
Iter: 24
Iter: 25
Iter: 26
Iter: 27
Iter: 28
Iter: 29
Iter: 30
Iter: 31
Iter: 32
Iter: 33
Iter: 34
Iter: 35
Iter: 36
Iter: 37
Iter: 38
Iter: 39
Iter: 40
Iter: 41
Iter: 42
Iter: 43
Iter: 44


c:\Users\bandarua\crda_gp\env\Lib\site-packages\gpytorch\models\exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(
C:\Users\bandarua\crda_gp\src\crda_gp\crda.py:36: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  x_new[perturb_idx] *= (1 + delta)


In [18]:
likelihood_aug_new = gpytorch.likelihoods.GaussianLikelihood()
model_aug_new = ARDExactGPModel_Dense(train_x=X_train_CRDA_tensor, 
                                      train_y=y_train_1_CRDA_tensor, 
                                      likelihood=likelihood_aug_new)

In [19]:
train_step(X_train=X_train_CRDA_tensor, 
           y_train=y_train_1_CRDA_tensor,
           model=model_aug_new,
           likelihood=likelihood_aug_new)

Iter: 10/100 | Loss: 0.709 | Noise: 0.312
Iter: 20/100 | Loss: 0.233 | Noise: 0.122
Iter: 30/100 | Loss: -0.285 | Noise: 0.044
Iter: 40/100 | Loss: -0.793 | Noise: 0.015
Iter: 50/100 | Loss: -1.215 | Noise: 0.005
Iter: 60/100 | Loss: -1.443 | Noise: 0.002
Iter: 70/100 | Loss: -1.500 | Noise: 0.002
Iter: 80/100 | Loss: -1.567 | Noise: 0.002
Iter: 90/100 | Loss: -1.612 | Noise: 0.002
Iter: 100/100 | Loss: -1.637 | Noise: 0.001


In [20]:
y_pred_aug_new = val_step(X_test=X_test_tensor, 
                      model=model_aug_new, 
                      likelihood=likelihood_aug_new)

In [21]:
MSE_aug_new = mean_squared_error(y_test_1_tensor, y_pred_aug_new)
MSE_aug_new

0.0011193822138011456

In [22]:
MSE

0.002021840075030923

In [23]:
X_train, X_test, y_train_2, y_test_2 = train_test_split(X, y_2, test_size=0.1, random_state=42)

# Turn data splits into tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_2_tensor = torch.tensor(y_train_2, dtype=torch.float32)
y_test_2_tensor = torch.tensor(y_test_2, dtype=torch.float32)

In [24]:
import gpytorch
from crda_gp.models import ARDExactGPModel_Dense
set_seed(42)
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ARDExactGPModel_Dense(train_x=X_train_tensor,
                              train_y=y_train_1_tensor,
                              likelihood=likelihood)
model

ARDExactGPModel_Dense(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): ConstantMean()
  (covar_module): ScaleKernel(
    (base_kernel): RBFKernel(
      (raw_lengthscale_constraint): Positive()
    )
    (raw_outputscale_constraint): Positive()
  )
)

In [25]:
import gpytorch
from crda_gp.models import ARDExactGPModel_Dense
set_seed(42)
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ARDExactGPModel_Dense(train_x=X_train_tensor,
                              train_y=y_train_2_tensor,
                              likelihood=likelihood)
model

ARDExactGPModel_Dense(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): ConstantMean()
  (covar_module): ScaleKernel(
    (base_kernel): RBFKernel(
      (raw_lengthscale_constraint): Positive()
    )
    (raw_outputscale_constraint): Positive()
  )
)

In [26]:
from crda_gp.utils import val_step
set_seed(42)
y_pred = val_step(X_test=X_test_tensor,
                  model=model,
                  likelihood=likelihood)

In [27]:
from sklearn.metrics import mean_squared_error
set_seed(42)
MSE_2 = mean_squared_error(y_true=y_test_2_tensor, 
                          y_pred=y_pred)

In [28]:
MSE_2

0.16506952047348022

In [29]:
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    pred_dist = likelihood(model(X_train_tensor))
    y_new_aug = pred_dist.mean

residual = y_train_2_tensor - y_new_aug
corr = abs(pd.DataFrame(X_train).corrwith(pd.Series(residual)))

preturb_idx= corr[abs(corr) < corr.mean().item()].sort_values(ascending=False).index.values
preturb_idx

c:\Users\bandarua\crda_gp\env\Lib\site-packages\gpytorch\models\exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(


array([5, 4, 1])

In [30]:
from crda_gp.crda import CRDA_function
X_train_CRDA_tensor, y_train_2_CRDA_tensor = CRDA_function(X_train_tensor,
                                                           y_train_2_tensor,
                                                           model=model,
                                                           likelihood=likelihood,
                                                           perturb_idx=preturb_idx,
                                                           p=0.1)

Starting the Loop
Iter: 0
Iter: 1
Iter: 2
Iter: 3
Iter: 4
Iter: 5
Iter: 6
Iter: 7
Iter: 8
Iter: 9
Iter: 10
Iter: 11
Iter: 12
Iter: 13
Iter: 14
Iter: 15
Iter: 16
Iter: 17
Iter: 18
Iter: 19
Iter: 20
Iter: 21
Iter: 22
Iter: 23
Iter: 24
Iter: 25
Iter: 26
Iter: 27
Iter: 28
Iter: 29
Iter: 30
Iter: 31
Iter: 32
Iter: 33
Iter: 34
Iter: 35
Iter: 36
Iter: 37
Iter: 38
Iter: 39
Iter: 40
Iter: 41
Iter: 42
Iter: 43
Iter: 44


In [31]:
likelihood_aug_new = gpytorch.likelihoods.GaussianLikelihood()
model_aug_new = ARDExactGPModel_Dense(train_x=X_train_CRDA_tensor, 
                                      train_y=y_train_2_CRDA_tensor, 
                                      likelihood=likelihood_aug_new)

In [32]:
train_step(X_train=X_train_CRDA_tensor, 
           y_train=y_train_2_CRDA_tensor,
           model=model_aug_new,
           likelihood=likelihood_aug_new)

Iter: 10/100 | Loss: 0.726 | Noise: 0.312
Iter: 20/100 | Loss: 0.276 | Noise: 0.122
Iter: 30/100 | Loss: -0.145 | Noise: 0.045
Iter: 40/100 | Loss: -0.434 | Noise: 0.017
Iter: 50/100 | Loss: -0.582 | Noise: 0.008
Iter: 60/100 | Loss: -0.662 | Noise: 0.006
Iter: 70/100 | Loss: -0.739 | Noise: 0.004
Iter: 80/100 | Loss: -0.770 | Noise: 0.003
Iter: 90/100 | Loss: -0.793 | Noise: 0.003
Iter: 100/100 | Loss: -0.826 | Noise: 0.003


In [33]:
y_pred_aug_new = val_step(X_test=X_test_tensor, 
                      model=model_aug_new, 
                      likelihood=likelihood_aug_new)

In [34]:
MSE_aug_new_2 = mean_squared_error(y_test_2_tensor, y_pred_aug_new)
MSE_aug_new_2

0.04895507171750069

In [35]:
MSE_2

0.16506952047348022